In [ ]:
import cv2
import numpy as np

# 1. Загрузка изображения
img = cv2.imread("9_final5.jpg")

# Уменьшаем изображение если оно слишком большое
max_size = 800
h, w = img.shape[:2]
if h > max_size or w > max_size:
    scale = max_size / max(h, w)
    img = cv2.resize(img, None, fx=scale, fy=scale)
    print(f"Изображение уменьшено до {img.shape[1]}x{img.shape[0]}")

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. Выбор 4 углов стола мышкой
points = []

def click_event(event, x, y, flags, param):
    global points, img_copy
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < 4:
        points.append([x, y])
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Выберите 4 угла стола", img_copy)

img_copy = img.copy()
cv2.imshow("Выберите 4 угла стола", img_copy)
cv2.setMouseCallback("Выберите 4 угла стола", click_event)
print("Выберите 4 угла стола")
cv2.waitKey(0)
cv2.destroyAllWindows()

# 3. Перспективная трансформация
src = np.float32(points)
width = int(max(np.linalg.norm(src[1] - src[0]), np.linalg.norm(src[3] - src[2])))
height = int(max(np.linalg.norm(src[2] - src[0]), np.linalg.norm(src[3] - src[1])))
dst = np.float32([[0, 0], [width, 0], [0, height], [width, height]])
M = cv2.getPerspectiveTransform(src, dst)
warped = cv2.warpPerspective(img, M, (width, height))

# 4. Обработка трансформированного изображения
gray_warped = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)

# Улучшенное размытие для удаления шума
blur = cv2.GaussianBlur(gray_warped, (9, 9), 0)

# Используем обычный threshold для черных объектов
_, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Морфологические операции для очистки шума и улучшения границ
kernel_small = np.ones((3, 3), np.uint8)
kernel_large = np.ones((5, 5), np.uint8)

# Удаляем мелкий шум
thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_small, iterations=2)

# Заполняем дыры внутри объектов
thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_large, iterations=3)

# Дополнительное сглаживание границ
thresh = cv2.dilate(thresh, kernel_small, iterations=1)
thresh = cv2.erode(thresh, kernel_small, iterations=1)

# 5. Поиск контуров
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Фильтруем контуры по площади (убираем слишком маленькие)
min_area = (width * height) * 0.01  # минимум 1% от площади изображения
contours = [c for c in contours if cv2.contourArea(c) > min_area]

# 6. Сортируем контуры по площади
contours = sorted(contours, key=cv2.contourArea, reverse=True)

print(f"Найдено контуров: {len(contours)}")



if len(contours) < 2:
    print("Ошибка: Найдено контуров меньше 2!")
    print("Попробуйте изменить освещение или контрастность объектов")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    exit()

# 7. Берем два самых больших контура
two_largest = contours[:2]

# Определяем который справа (контейнер) и который слева (объект)
moments1 = cv2.moments(two_largest[0])
moments2 = cv2.moments(two_largest[1])
center1_x = int(moments1['m10'] / moments1['m00'])
center2_x = int(moments2['m10'] / moments2['m00'])

# Справа - контейнер, слева - объект
if center1_x > center2_x:
    container_contour = two_largest[0]
    object_contour = two_largest[1]
else:
    container_contour = two_largest[1]
    object_contour = two_largest[0]

# 8. Получаем минимальные ограничивающие прямоугольники
container_rect = cv2.minAreaRect(container_contour)
object_rect = cv2.minAreaRect(object_contour)

# 9. Размеры контейнера и объекта
container_width = max(container_rect[1])
container_height = min(container_rect[1])
object_width = max(object_rect[1])
object_height = min(object_rect[1])

# 10. Проверка: помещается ли объект в контейнер
# Учитываем возможность поворота объекта на 90 градусов
fits = False
if (object_width <= container_width and object_height <= container_height):
    fits = True
elif (object_height <= container_width and object_width <= container_height):
    fits = True

# 11. Визуализация - границы контейнера и объекта
result_img = warped.copy()

# Рисуем границы контейнера (зеленый)
cv2.drawContours(result_img, [container_contour], 0, (0, 255, 0), 3)

# Рисуем границы объекта (синий)
cv2.drawContours(result_img, [object_contour], 0, (255, 0, 0), 3)

# 12. Вывод результата
result_text = "ПОМЕЩАЕТСЯ" if fits else "НЕ ПОМЕЩАЕТСЯ"

print(f"\n{'='*50}")
print(f"РЕЗУЛЬТАТ: {result_text}")
print(f"{'='*50}")
print(f"Контейнер: {container_width:.1f} x {container_height:.1f} пикселей")
print(f"Объект: {object_width:.1f} x {object_height:.1f} пикселей")
print(f"{'='*50}\n")

# 13. Показываем результаты
cv2.imshow("Исходное изображение", img)
cv2.imshow("Вид сверху", warped)
cv2.imshow("Результат", result_img)



cv2.waitKey(0)
cv2.destroyAllWindows()

Изображение уменьшено до 600x800
Выберите 4 угла стола
Найдено контуров: 2

РЕЗУЛЬТАТ: НЕ ПОМЕЩАЕТСЯ
Контейнер: 125.3 x 75.9 пикселей
Объект: 206.0 x 189.0 пикселей

Нажмите любую клавишу для выхода...
